# V4 — Improved Audio Deepfake Detection
## Target: EER < 5% on ASVspoof 2019 LA Eval

**Key improvements over v3 (10.67% EER):**
1. Deeper LCNN with Squeeze-and-Excitation (SE) attention blocks
2. Feature-level augmentation (Gaussian noise + random feature masking)
3. Multi-seed ensemble (3 models, score averaging)
4. Extended training (80 epochs) with cosine warm restarts
5. Gradient accumulation for effective larger batch size
6. Better weight decay schedule
7. Score calibration on dev set

In [ ]:
# Cell 1: Install dependencies
!pip install PyWavelets tqdm librosa frft --quiet
print("Done!")

In [ ]:
# Cell 2: Imports and FrFT setup
import os, warnings, time, copy, gc
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import soundfile as sf
import pywt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_curve, balanced_accuracy_score
from scipy.optimize import brentq
from scipy.interpolate import interp1d
import matplotlib.pyplot as plt
from tqdm import tqdm

# ── FrFT: Fractional Fourier Transform ──────────────────────
try:
    from frft import frft as _ext_frft
    def compute_frft(signal, alpha):
        return _ext_frft(np.asarray(signal, dtype=complex), alpha)
    print("✓ frft package loaded (proper DFrFT)")
except Exception:
    def compute_frft(signal, alpha):
        N   = len(signal)
        x   = np.asarray(signal, dtype=complex)
        if np.isclose(alpha % 2, 0): return x.copy()
        if np.isclose(alpha % 2, 1): return np.fft.fft(x) / np.sqrt(N)
        phi   = alpha * np.pi / 2
        n     = np.arange(N, dtype=float)
        chirp = np.exp(1j * np.pi * np.tan(phi / 2) * n**2 / N)
        return np.fft.fft(x * chirp) / np.sqrt(N)
    print("⚠  Using built-in FrFT (frft package unavailable)")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✓ Device: {device}")
if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Cell 3: Dataset paths
BASE = '/kaggle/input/datasets/awsaf49/asvpoof-2019-dataset'

key_paths = [
    'LA/LA/ASVspoof2019_LA_train/flac',
    'LA/LA/ASVspoof2019_LA_dev/flac',
    'LA/LA/ASVspoof2019_LA_eval/flac',
    'LA/LA/ASVspoof2019_LA_cm_protocols',
]

print("Dataset structure:")
for p in key_paths:
    full = os.path.join(BASE, p)
    if os.path.isdir(full):
        count = sum(1 for _ in os.scandir(full))
        print(f"  ✓ {p}/  [{count:,} files]")
    else:
        print(f"  ✗ {p}/  [NOT FOUND]")

In [ ]:
# Cell 4: Paths and parameters
BASE = '/kaggle/input/datasets/awsaf49/asvpoof-2019-dataset'
LA   = BASE + '/LA/LA'

PROTOCOLS   = LA + '/ASVspoof2019_LA_cm_protocols'
TRAIN_PROTO = PROTOCOLS + '/ASVspoof2019.LA.cm.train.trn.txt'
DEV_PROTO   = PROTOCOLS + '/ASVspoof2019.LA.cm.dev.trl.txt'
EVAL_PROTO  = PROTOCOLS + '/ASVspoof2019.LA.cm.eval.trl.txt'

TRAIN_AUDIO = LA + '/ASVspoof2019_LA_train/flac'
DEV_AUDIO   = LA + '/ASVspoof2019_LA_dev/flac'
EVAL_AUDIO  = LA + '/ASVspoof2019_LA_eval/flac'

# Verify paths
for label, path in [
    ('Train protocol', TRAIN_PROTO), ('Dev   protocol', DEV_PROTO),
    ('Eval  protocol', EVAL_PROTO),  ('Train audio',    TRAIN_AUDIO),
    ('Dev   audio',    DEV_AUDIO),   ('Eval  audio',    EVAL_AUDIO)
]:
    ok = os.path.exists(path)
    print(f"{'✓' if ok else '✗'}  {label}: {path}")

# ── Feature parameters ──────────────────────────────────────
SR       = 16000
MAX_LEN  = 96000                                        # 6 seconds
ALPHAS   = [0.05, 0.15, 0.3, 0.5, 0.7, 0.85, 0.95]   # 7 FrFT angles
WPD_LVL  = 5                                           # 32 sub-bands
WAVELET  = 'db4'

# Feature dimensions
N_FRFT      = 7 * 8 * 8       # 448
N_WPD       = 32 * 8          # 256
N_FEAT_USED = N_FRFT + N_WPD  # 704

# ── Training parameters (IMPROVED) ──────────────────────────
BATCH        = 256
ACCUM_STEPS  = 2              # effective batch = 512
EPOCHS       = 80
LR           = 3e-4
WEIGHT_DECAY = 1e-3           # stronger L2
SEED_LIST    = [42, 2024, 7]  # 3 seeds for ensemble
PATIENCE     = 15

print(f"\nFeatures: FrFT({N_FRFT}) + WPD({N_WPD}) = {N_FEAT_USED}")
print(f"Training: batch={BATCH}, accum={ACCUM_STEPS}, effective={BATCH*ACCUM_STEPS}")
print(f"Epochs: {EPOCHS}, patience: {PATIENCE}, seeds: {SEED_LIST}")

In [ ]:
# Cell 5: Load protocols
def load_protocol(proto_path, audio_dir):
    df = pd.read_csv(
        proto_path, sep=' ', header=None,
        names=['speaker', 'file', 'env', 'attack', 'label']
    )
    df['binary'] = (df['label'] == 'bonafide').astype(int)
    df['path']   = df['file'].apply(
        lambda f: os.path.join(audio_dir, f + '.flac')
    )
    return df

train_df = load_protocol(TRAIN_PROTO, TRAIN_AUDIO)
dev_df   = load_protocol(DEV_PROTO,   DEV_AUDIO)
eval_df  = load_protocol(EVAL_PROTO,  EVAL_AUDIO)

print("Dataset split summary:")
print(f"{'Split':<8} {'Total':>8} {'Bonafide':>10} {'Spoof':>8}")
print("─" * 38)
for name, df in [('Train', train_df), ('Dev', dev_df), ('Eval', eval_df)]:
    n_real  = (df['binary'] == 1).sum()
    n_spoof = (df['binary'] == 0).sum()
    print(f"{name:<8} {len(df):>8,} {n_real:>10,} {n_spoof:>8,}")

In [ ]:
# Cell 6: Feature extraction functions
def load_audio(path, max_len=MAX_LEN):
    signal, _ = sf.read(path)
    if signal.ndim > 1:
        signal = signal.mean(axis=1)
    if len(signal) >= max_len:
        signal = signal[:max_len]
    else:
        signal = np.pad(signal, (0, max_len - len(signal)))
    return signal.astype(np.float32)

def stats8(arr):
    arr  = np.abs(arr).astype(np.float64) + 1e-10
    p    = arr / arr.sum()
    return np.array([
        arr.mean(), arr.std(), arr.max(), arr.min(),
        np.percentile(arr, 25), np.percentile(arr, 75),
        np.sum(arr**2),
        -np.sum(p * np.log(p + 1e-10))
    ], dtype=np.float32)

def extract_frft_features(signal, alphas=ALPHAS):
    feats = []
    for alpha in alphas:
        transformed = compute_frft(signal.astype(complex), alpha)
        mag = np.abs(transformed)
        chunks = np.array_split(mag, 8)
        for chunk in chunks:
            feats.append(stats8(chunk))
    return np.concatenate(feats)

def extract_wpd_features(signal, wavelet=WAVELET, level=WPD_LVL):
    wp    = pywt.WaveletPacket(data=signal, wavelet=wavelet,
                               mode='symmetric', maxlevel=level)
    nodes = wp.get_level(level, 'natural')
    feats = [stats8(node.data) for node in nodes]
    return np.concatenate(feats)

def extract_features(signal):
    frft_feats = extract_frft_features(signal)  # 448
    wpd_feats  = extract_wpd_features(signal)   # 256
    combined = np.concatenate([frft_feats, wpd_feats])
    combined = np.nan_to_num(combined, nan=0.0, posinf=0.0, neginf=0.0)
    return combined.astype(np.float32)

# Sanity check
test_sig = load_audio(train_df.iloc[0]['path'])
test_feat = extract_features(test_sig)
print(f"Feature vector: {test_feat.shape}")  # (704,)
assert test_feat.shape == (N_FEAT_USED,)
print("✓ Feature engineering ready!")

In [ ]:
# Cell 7: Extract features (with caching + checkpointing)
FEAT_DIR = '/kaggle/working/features_v4'
os.makedirs(FEAT_DIR, exist_ok=True)

def process_split(df, split_name):
    path_X       = os.path.join(FEAT_DIR, f'{split_name}_X.npy')
    path_y       = os.path.join(FEAT_DIR, f'{split_name}_y.npy')
    path_partial = os.path.join(FEAT_DIR, f'{split_name}_partial.npz')

    if os.path.exists(path_X) and os.path.exists(path_y):
        X = np.load(path_X)
        y = np.load(path_y)
        if X.ndim == 2 and X.shape[1] == N_FEAT_USED and len(X) == len(y):
            print(f"[{split_name}] ✓ Loaded from cache → X:{X.shape}  y:{y.shape}")
            return X, y
        else:
            os.remove(path_X); os.remove(path_y)

    start_idx = 0
    X_list, y_list = [], []

    if os.path.exists(path_partial):
        data = np.load(path_partial)
        X_list = list(data['X'])
        y_list = list(data['y'])
        start_idx = int(data['next_idx'])
        print(f"[{split_name}] ↻ Resuming from {start_idx:,} / {len(df):,}")
    else:
        print(f"[{split_name}] Extracting features for {len(df):,} files...")

    skipped = 0
    for idx in tqdm(range(start_idx, len(df)), desc=split_name,
                    initial=start_idx, total=len(df)):
        row = df.iloc[idx]
        try:
            signal = load_audio(row['path'])
            feat = extract_features(signal)
            X_list.append(feat)
            y_list.append(row['binary'])
        except Exception:
            skipped += 1

        if (idx + 1) % 5000 == 0:
            np.savez(path_partial,
                     X=np.array(X_list, dtype=np.float32),
                     y=np.array(y_list, dtype=np.int64),
                     next_idx=np.array([idx + 1]))

    X = np.array(X_list, dtype=np.float32)
    y = np.array(y_list, dtype=np.int64)

    np.save(path_X, X)
    np.save(path_y, y)
    if os.path.exists(path_partial):
        os.remove(path_partial)

    print(f"[{split_name}] ✓ Done → X:{X.shape}  y:{y.shape}  skipped:{skipped}")
    return X, y

# Also check if v2 features (996-dim) exist and we can slice them
V2_FEAT_DIR = '/kaggle/working/features_v2'
if os.path.exists(os.path.join(V2_FEAT_DIR, 'train_X.npy')):
    print("Found v2 features (996-dim) — slicing to 704...")
    for split in ['train', 'dev', 'eval']:
        X_full = np.load(os.path.join(V2_FEAT_DIR, f'{split}_X.npy'))
        y_full = np.load(os.path.join(V2_FEAT_DIR, f'{split}_y.npy'))
        X_cut = X_full[:, :N_FEAT_USED]
        np.save(os.path.join(FEAT_DIR, f'{split}_X.npy'), X_cut)
        np.save(os.path.join(FEAT_DIR, f'{split}_y.npy'), y_full)
        print(f"  {split}: {X_full.shape} → {X_cut.shape}")

t0 = time.time()
X_train, y_train = process_split(train_df, 'train')
X_dev,   y_dev   = process_split(dev_df,   'dev')
X_eval,  y_eval  = process_split(eval_df,  'eval')
elapsed = (time.time() - t0) / 60

print(f"\nAll done in {elapsed:.1f} min")
print(f"  X_train: {X_train.shape}  X_dev: {X_dev.shape}  X_eval: {X_eval.shape}")

In [ ]:
# Cell 8: Dataset and DataLoaders with feature augmentation

# StandardScaler
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_dev_sc   = scaler.transform(X_dev)
X_eval_sc  = scaler.transform(X_eval)

class AudioDataset(Dataset):
    def __init__(self, X, y, augment=False):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
        self.augment = augment

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        x = self.X[i]
        if self.augment and self.y[i].item() == 0:  # augment spoof samples more
            # Gaussian noise (small, preserves phase artifacts)
            if torch.rand(1).item() < 0.3:
                x = x + torch.randn_like(x) * 0.05
            # Random feature masking (like SpecAugment but for features)
            if torch.rand(1).item() < 0.3:
                mask_len = torch.randint(10, 50, (1,)).item()
                mask_start = torch.randint(0, max(1, len(x) - mask_len), (1,)).item()
                x[mask_start:mask_start + mask_len] = 0
        elif self.augment:
            # Lighter augmentation for bonafide
            if torch.rand(1).item() < 0.15:
                x = x + torch.randn_like(x) * 0.02
        return x, self.y[i]

# Training set WITH augmentation
train_ds = AudioDataset(X_train_sc, y_train, augment=True)
dev_ds   = AudioDataset(X_dev_sc,   y_dev,   augment=False)
eval_ds  = AudioDataset(X_eval_sc,  y_eval,  augment=False)

train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=True)
dev_loader   = DataLoader(dev_ds,   batch_size=512, shuffle=False,
                          num_workers=2, pin_memory=True)
eval_loader  = DataLoader(eval_ds,  batch_size=512, shuffle=False,
                          num_workers=2, pin_memory=True)

# Class weights
n_total = len(y_train)
n_real  = int(y_train.sum())
n_spoof = n_total - n_real
class_weight = torch.tensor([
    n_total / (2.0 * n_spoof),
    n_total / (2.0 * n_real)
], dtype=torch.float32).to(device)

print(f"Class weights → spoof: {class_weight[0]:.3f} | bonafide: {class_weight[1]:.3f}")
print(f"Train batches: {len(train_loader)} × {BATCH} = {len(train_loader)*BATCH:,}")
print(f"Effective batch: {BATCH * ACCUM_STEPS}")
print("✓ Datasets ready!")

In [ ]:
# Cell 9: Improved Model Architecture — LCNN v4 with SE Attention

class MFM(nn.Module):
    """Max Feature Map: competitive activation that halves channels."""
    def forward(self, x):
        n = x.shape[1] // 2
        return torch.max(x[:, :n], x[:, n:])


class SEBlock(nn.Module):
    """Squeeze-and-Excitation block for channel attention."""
    def __init__(self, channels, reduction=4):
        super().__init__()
        self.fc1 = nn.Linear(channels, channels // reduction)
        self.fc2 = nn.Linear(channels // reduction, channels)

    def forward(self, x):
        # x: (batch, channels)
        se = F.relu(self.fc1(x))
        se = torch.sigmoid(self.fc2(se))
        return x * se  # channel-wise reweighting


class LCNNBlock(nn.Module):
    """Single LCNN block: Linear → MFM → BN → SE → Dropout + residual."""
    def __init__(self, in_dim, hidden_dim, dropout=0.3, use_se=True):
        super().__init__()
        out_dim = hidden_dim // 2  # MFM halves
        self.fc  = nn.Linear(in_dim, hidden_dim)
        self.mfm = MFM()
        self.bn  = nn.BatchNorm1d(out_dim)
        self.se  = SEBlock(out_dim) if use_se else nn.Identity()
        self.drop = nn.Dropout(dropout)

        # Residual projection if dimensions differ
        self.residual = nn.Linear(in_dim, out_dim) if in_dim != out_dim else nn.Identity()

    def forward(self, x):
        identity = self.residual(x)
        out = self.fc(x)
        out = self.mfm(out)
        out = self.bn(out)
        out = self.se(out)
        out = self.drop(out)
        out = out + identity
        return out


class LCNN_v4(nn.Module):
    """
    Light CNN v4 — Deeper architecture with SE attention.

    Architecture:
        Input(704) → Dropout(0.1)
        → Block1: 704→1024→512 (MFM) + SE + residual
        → Block2: 512→1024→512 (MFM) + SE + residual  [same dim, deeper]
        → Block3: 512→512→256 (MFM) + SE + residual
        → Block4: 256→256→128 (MFM) + SE + residual
        → Block5: 128→128→64  (MFM) + residual
        → Classifier: 64→2

    Total params: ~1.1M (still lightweight)
    """
    def __init__(self, input_dim):
        super().__init__()
        self.input_drop = nn.Dropout(0.15)

        self.block1 = LCNNBlock(input_dim, 1024, dropout=0.5, use_se=True)
        self.block2 = LCNNBlock(512,       1024, dropout=0.45, use_se=True)  # deeper
        self.block3 = LCNNBlock(512,        512, dropout=0.4, use_se=True)
        self.block4 = LCNNBlock(256,        256, dropout=0.3, use_se=True)
        self.block5 = LCNNBlock(128,        128, dropout=0.2, use_se=False)

        self.classifier = nn.Linear(64, 2)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.input_drop(x)
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        x = self.block5(x)
        return self.classifier(x)


# Also keep the v3 architecture for comparison
class LCNN_v3(nn.Module):
    """Original v3 architecture (576K params) for ablation."""
    def __init__(self, input_dim):
        super().__init__()
        self.input_drop = nn.Dropout(0.1)
        self.block1 = nn.Sequential(
            nn.Linear(input_dim, 512), MFM(),
            nn.BatchNorm1d(256), nn.Dropout(0.5)
        )
        self.fc2   = nn.Linear(256, 512)
        self.mfm2  = MFM()
        self.bn2   = nn.BatchNorm1d(256)
        self.drop2 = nn.Dropout(0.4)
        self.block3 = nn.Sequential(
            nn.Linear(256, 256), MFM(),
            nn.BatchNorm1d(128), nn.Dropout(0.3)
        )
        self.block4 = nn.Sequential(
            nn.Linear(128, 128), MFM(),
            nn.BatchNorm1d(64), nn.Dropout(0.25)
        )
        self.classifier = nn.Linear(64, 2)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.input_drop(x)
        x = self.block1(x)
        identity = x
        x = self.drop2(self.bn2(self.mfm2(self.fc2(x))))
        x = x + identity
        x = self.block3(x)
        x = self.block4(x)
        return self.classifier(x)


# Print architectures
for name, ModelClass in [('LCNN_v3', LCNN_v3), ('LCNN_v4', LCNN_v4)]:
    m = ModelClass(N_FEAT_USED)
    n_params = sum(p.numel() for p in m.parameters())
    print(f"\n{name}: {n_params:,} parameters")
    if name == 'LCNN_v4':
        print(m)

In [ ]:
# Cell 10: Training and evaluation functions

def compute_eer(labels, scores):
    """Equal Error Rate: FAR == FRR. Lower = better."""
    fpr, tpr, thresholds = roc_curve(labels, scores, pos_label=1)
    fnr = 1 - tpr
    # Use interpolation for more accurate EER
    try:
        eer = brentq(lambda x: interp1d(fpr, fnr)(x) - x, 0, 1)
        thresh = float(interp1d(fpr, thresholds)(eer))
    except:
        idx = np.nanargmin(np.abs(fnr - fpr))
        eer = float(np.mean([fpr[idx], fnr[idx]]))
        thresh = float(thresholds[idx])
    return eer * 100, thresh

def evaluate(mdl, loader, return_scores=False):
    """Return EER (%), accuracy (%), and optionally raw scores."""
    mdl.eval()
    all_labels, all_scores = [], []
    correct = 0
    with torch.no_grad():
        for X, y in loader:
            X, y    = X.to(device), y.to(device)
            logits  = mdl(X)
            probs   = torch.softmax(logits, dim=1)[:, 1]
            correct += (logits.argmax(1) == y).sum().item()
            all_labels.extend(y.cpu().numpy())
            all_scores.extend(probs.cpu().numpy())
    acc = 100.0 * correct / len(all_labels)
    labels = np.array(all_labels)
    scores = np.array(all_scores)
    eer, thresh = compute_eer(labels, scores)
    bal_acc = balanced_accuracy_score(labels, (scores >= thresh).astype(int)) * 100
    if return_scores:
        return eer, acc, bal_acc, scores, labels
    return eer, acc, bal_acc

def set_seed(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

print("✓ Training functions ready")

In [ ]:
# Cell 11: Train a single model

def train_single_model(model_class, seed, tag, epochs=EPOCHS, patience=PATIENCE):
    """Train one model with given seed. Returns model, history, eval results."""
    set_seed(seed)
    print(f"\n{'='*65}")
    print(f"  Training {tag} (seed={seed})")
    print(f"{'='*65}")

    model = model_class(input_dim=N_FEAT_USED).to(device)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"  Parameters: {n_params:,}")

    criterion = nn.CrossEntropyLoss(
        weight=class_weight,
        label_smoothing=0.1
    )
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY,
        betas=(0.9, 0.999)
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=10, T_mult=2, eta_min=1e-6
    )

    history = {'loss': [], 'dev_eer': [], 'dev_acc': [], 'dev_bal_acc': []}
    best_dev_eer = float('inf')
    best_epoch = 0
    patience_counter = 0
    best_state = None

    save_path = f'/kaggle/working/best_{tag}.pt'

    print(f"  Loss: CrossEntropy + label_smoothing=0.1")
    print(f"  Optimizer: AdamW (lr={LR}, wd={WEIGHT_DECAY})")
    print(f"  Scheduler: CosineAnnealingWarmRestarts(T_0=10, T_mult=2)")
    print(f"  Gradient accumulation: {ACCUM_STEPS} steps")
    print(f"  Max epochs: {epochs}, patience: {patience}")
    print(f"\n{'Epoch':>6} | {'Loss':>8} | {'Dev EER':>9} | {'Dev Acc':>8} | {'Bal Acc':>8} | {'LR':>10} | {'Time':>6}")
    print("─" * 75)

    for ep in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        t0 = time.time()
        optimizer.zero_grad()

        for batch_idx, (X, y) in enumerate(train_loader):
            X, y = X.to(device), y.to(device)
            logits = model(X)
            loss = criterion(logits, y) / ACCUM_STEPS
            loss.backward()

            if (batch_idx + 1) % ACCUM_STEPS == 0 or (batch_idx + 1) == len(train_loader):
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=3.0)
                optimizer.step()
                optimizer.zero_grad()

            total_loss += loss.item() * ACCUM_STEPS

        scheduler.step()

        avg_loss = total_loss / len(train_loader)
        dev_eer, dev_acc, dev_bal_acc = evaluate(model, dev_loader)
        elapsed = time.time() - t0
        current_lr = optimizer.param_groups[0]['lr']

        history['loss'].append(avg_loss)
        history['dev_eer'].append(dev_eer)
        history['dev_acc'].append(dev_acc)
        history['dev_bal_acc'].append(dev_bal_acc)

        flag = ''
        if dev_eer < best_dev_eer:
            best_dev_eer = dev_eer
            best_epoch = ep
            patience_counter = 0
            flag = '  ← best'
            best_state = copy.deepcopy(model.state_dict())
            torch.save(best_state, save_path)
        else:
            patience_counter += 1

        print(f"{ep:>6} | {avg_loss:>8.4f} | {dev_eer:>8.2f}% | {dev_acc:>7.2f}% | {dev_bal_acc:>7.2f}% | {current_lr:>10.2e} | {elapsed:>5.1f}s{flag}")

        if patience_counter >= patience:
            print(f"\n⚡ Early stopping at epoch {ep}")
            break

    # Load best and evaluate
    model.load_state_dict(best_state)
    eval_eer, eval_acc, eval_bal_acc, eval_scores, eval_labels = evaluate(
        model, eval_loader, return_scores=True
    )

    print(f"\n✓ Best Dev EER: {best_dev_eer:.2f}% at epoch {best_epoch}")
    print(f"✓ Eval EER: {eval_eer:.2f}% | Acc: {eval_acc:.2f}% | Bal Acc: {eval_bal_acc:.2f}%")
    print(f"✓ Saved → {save_path}")

    return model, history, eval_eer, eval_acc, eval_bal_acc, eval_scores, eval_labels

print("✓ Training function ready")

In [ ]:
# Cell 12: Train 3-model ensemble with LCNN_v4

ensemble_results = []
all_eval_scores = []
all_dev_scores = []

for i, seed in enumerate(SEED_LIST):
    tag = f"lcnn_v4_seed{seed}"
    model, history, eval_eer, eval_acc, eval_bal_acc, eval_scores, eval_labels = \
        train_single_model(LCNN_v4, seed, tag)

    ensemble_results.append({
        'seed': seed,
        'tag': tag,
        'eval_eer': eval_eer,
        'eval_acc': eval_acc,
        'eval_bal_acc': eval_bal_acc,
        'history': history
    })
    all_eval_scores.append(eval_scores)

    # Also get dev scores for calibration
    _, _, _, dev_scores, dev_labels = evaluate(model, dev_loader, return_scores=True)
    all_dev_scores.append(dev_scores)

    # Free GPU memory
    del model
    torch.cuda.empty_cache()
    gc.collect()

print(f"\n{'='*65}")
print("  Individual Model Results")
print(f"{'='*65}")
for r in ensemble_results:
    print(f"  Seed {r['seed']}: EER={r['eval_eer']:.2f}%  Acc={r['eval_acc']:.2f}%  Bal={r['eval_bal_acc']:.2f}%")

In [ ]:
# Cell 13: Ensemble score fusion + evaluation

# Average scores across all models
ensemble_eval_scores = np.mean(all_eval_scores, axis=0)
ensemble_dev_scores  = np.mean(all_dev_scores, axis=0)

# Evaluate ensemble on eval
ensemble_eer, ensemble_thresh = compute_eer(eval_labels, ensemble_eval_scores)
ensemble_preds = (ensemble_eval_scores >= ensemble_thresh).astype(int)
ensemble_acc = 100.0 * np.mean(ensemble_preds == eval_labels)
ensemble_bal_acc = balanced_accuracy_score(eval_labels, ensemble_preds) * 100

# Also calibrate threshold on dev set
dev_eer, dev_thresh = compute_eer(dev_labels, ensemble_dev_scores)
# Use dev threshold on eval
calibrated_preds = (ensemble_eval_scores >= dev_thresh).astype(int)
calibrated_acc = 100.0 * np.mean(calibrated_preds == eval_labels)
calibrated_bal_acc = balanced_accuracy_score(eval_labels, calibrated_preds) * 100

print("="*65)
print("  ENSEMBLE RESULTS (Score Averaging)")
print("="*65)
print(f"\n  Ensemble Eval EER      : {ensemble_eer:.2f}%")
print(f"  Ensemble Eval Accuracy : {ensemble_acc:.2f}%")
print(f"  Ensemble Bal Accuracy  : {ensemble_bal_acc:.2f}%")
print(f"\n  With dev-calibrated threshold ({dev_thresh:.4f}):")
print(f"    Accuracy  : {calibrated_acc:.2f}%")
print(f"    Bal Acc   : {calibrated_bal_acc:.2f}%")
print(f"\n  Comparison:")
print(f"    v1 (original, 448 feats)       : 11.79% EER")
print(f"    v2 (996 feats, overfit)         : 13.55% EER")
print(f"    v3 (704 feats, single model)    : 10.67% EER")
for r in ensemble_results:
    print(f"    v4 (seed {r['seed']}, individual)    : {r['eval_eer']:.2f}% EER")
print(f"    v4 ENSEMBLE ({len(SEED_LIST)} models)       : {ensemble_eer:.2f}% EER  ← NEW")
print("="*65)

In [ ]:
# Cell 14: Also train v3 architecture with same improvements for fair comparison

print("Training v3 architecture with v4 training improvements...")
v3_model, v3_history, v3_eer, v3_acc, v3_bal_acc, v3_scores, v3_labels = \
    train_single_model(LCNN_v3, seed=42, tag='lcnn_v3_improved', epochs=60, patience=15)

print(f"\nv3 (original training): 10.67% EER")
print(f"v3 (improved training): {v3_eer:.2f}% EER")
print(f"v4 ensemble:           {ensemble_eer:.2f}% EER")

# Free GPU
del v3_model
torch.cuda.empty_cache()
gc.collect()

In [ ]:
# Cell 15: Training curves visualization

fig, axes = plt.subplots(2, 3, figsize=(20, 12))
fig.suptitle(f'V4 Training Curves — Ensemble EER = {ensemble_eer:.2f}%', fontsize=14)

colors = ['#2196F3', '#4CAF50', '#FF9800']

# Row 1: Individual model curves
for i, r in enumerate(ensemble_results):
    h = r['history']
    ep_range = range(1, len(h['loss']) + 1)
    c = colors[i % len(colors)]

    axes[0][0].plot(ep_range, h['loss'], f'-', color=c, lw=1.5, label=f"Seed {r['seed']}")
    axes[0][1].plot(ep_range, h['dev_eer'], f'-', color=c, lw=1.5, label=f"Seed {r['seed']}")
    axes[0][2].plot(ep_range, h['dev_acc'], f'-', color=c, lw=1.5, label=f"Seed {r['seed']}")

axes[0][0].set_title('Training Loss'); axes[0][0].set_ylabel('Loss')
axes[0][0].legend(); axes[0][0].grid(True, alpha=0.3)

axes[0][1].set_title('Dev EER (%)'); axes[0][1].set_ylabel('EER (%)')
axes[0][1].axhline(ensemble_eer, color='red', ls='--', lw=2, label=f'Ensemble Eval EER={ensemble_eer:.2f}%')
axes[0][1].legend(); axes[0][1].grid(True, alpha=0.3)

axes[0][2].set_title('Dev Accuracy (%)'); axes[0][2].set_ylabel('Accuracy (%)')
axes[0][2].legend(); axes[0][2].grid(True, alpha=0.3)

# Row 2: Comparison plots
# Version comparison bar chart
versions = ['v1\n(448)', 'v2\n(996)', 'v3\n(704)', 'v3+\n(improved)', 'v4\n(single best)', f'v4\n(ensemble)']
eers = [11.79, 13.55, 10.67, v3_eer,
        min(r['eval_eer'] for r in ensemble_results), ensemble_eer]
bar_colors = ['#90CAF9', '#EF9A9A', '#A5D6A7', '#CE93D8', '#FFE082', '#4CAF50']

bars = axes[1][0].bar(versions, eers, color=bar_colors, edgecolor='white', width=0.6)
for bar, eer in zip(bars, eers):
    axes[1][0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.15,
                    f'{eer:.2f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')
axes[1][0].set_ylabel('EER (%) — lower is better')
axes[1][0].set_title('Version Comparison')
axes[1][0].set_ylim(0, max(eers) * 1.3)
axes[1][0].grid(axis='y', alpha=0.3)

# Score distribution
axes[1][1].hist(ensemble_eval_scores[eval_labels == 1], bins=100, alpha=0.7,
                color='steelblue', label='Bonafide', density=True)
axes[1][1].hist(ensemble_eval_scores[eval_labels == 0], bins=100, alpha=0.7,
                color='tomato', label='Spoof', density=True)
axes[1][1].axvline(ensemble_thresh, color='green', ls='--', lw=2,
                   label=f'EER threshold={ensemble_thresh:.3f}')
axes[1][1].set_title('Ensemble Score Distribution')
axes[1][1].set_xlabel('Bonafide Probability')
axes[1][1].legend(); axes[1][1].grid(True, alpha=0.3)

# ROC curve
fpr, tpr, _ = roc_curve(eval_labels, ensemble_eval_scores, pos_label=1)
axes[1][2].plot(fpr, tpr, 'b-', lw=2, label=f'v4 Ensemble (EER={ensemble_eer:.2f}%)')
axes[1][2].plot([0, 1], [0, 1], 'k--', alpha=0.3)
axes[1][2].set_xlabel('False Positive Rate')
axes[1][2].set_ylabel('True Positive Rate')
axes[1][2].set_title('ROC Curve')
axes[1][2].legend(); axes[1][2].grid(True, alpha=0.3)

for ax in axes.flat:
    ax.set_xlabel(ax.get_xlabel() or 'Epoch')

plt.tight_layout()
plt.savefig('/kaggle/working/v4_training_curves.png', dpi=150)
plt.show()
print("Saved → /kaggle/working/v4_training_curves.png")

In [ ]:
# Cell 16: Per-attack analysis (important for understanding where improvements come from)

# Load eval protocol to get attack types
eval_proto_df = load_protocol(EVAL_PROTO, EVAL_AUDIO)

# Map scores back to protocol
# Note: scores are in same order as eval_df which matches eval_loader order
assert len(ensemble_eval_scores) == len(eval_proto_df)

eval_proto_df['score'] = ensemble_eval_scores
eval_proto_df['pred_label'] = (ensemble_eval_scores >= ensemble_thresh).astype(int)

print("Per-Attack Analysis (Ensemble v4)")
print("="*70)
print(f"{'Attack':<12} {'Count':>8} {'Correct':>10} {'Accuracy':>10} {'Avg Score':>10}")
print("─"*70)

for attack in sorted(eval_proto_df['attack'].unique()):
    subset = eval_proto_df[eval_proto_df['attack'] == attack]
    correct = (subset['pred_label'] == subset['binary']).sum()
    acc = 100.0 * correct / len(subset)
    avg_score = subset['score'].mean()
    print(f"{attack:<12} {len(subset):>8,} {correct:>10,} {acc:>9.2f}% {avg_score:>9.4f}")

print("─"*70)
total_correct = (eval_proto_df['pred_label'] == eval_proto_df['binary']).sum()
print(f"{'TOTAL':<12} {len(eval_proto_df):>8,} {total_correct:>10,} {100.0*total_correct/len(eval_proto_df):>9.2f}%")

In [ ]:
# Cell 17: Final Summary

print("═"*65)
print("  CAPSTONE PROJECT — V4 RESULTS SUMMARY")
print("═"*65)
print(f"\n  Task      : Deepfake Audio Detection (LA subset)")
print(f"  Dataset   : ASVspoof 2019")
print(f"  Train     : {len(y_train):,} | Dev: {len(y_dev):,} | Eval: {len(y_eval):,}")
print(f"\n  Feature Engineering:")
print(f"    FrFT  7 angles × 8 chunks × 8 stats  = {N_FRFT:>3} features")
print(f"    WPD   32 sub-bands × 8 stats          = {N_WPD:>3} features")
print(f"    Fused total                           = {N_FEAT_USED:>3} features")

print(f"\n  Version History:")
print(f"    v1 (original, 448 feats)               Eval EER: 11.79%")
print(f"    v2 (996 feats, overfit)                 Eval EER: 13.55%")
print(f"    v3 (704 feats, LCNN)                    Eval EER: 10.67%")
print(f"    v3+ (improved training)                 Eval EER: {v3_eer:.2f}%")
for r in ensemble_results:
    print(f"    v4 (LCNN_v4, seed {r['seed']})              Eval EER: {r['eval_eer']:.2f}%")
print(f"    v4 ENSEMBLE ({len(SEED_LIST)} models, avg)         Eval EER: {ensemble_eer:.2f}%  ← BEST")

best_single_v4 = min(r['eval_eer'] for r in ensemble_results)
print(f"\n  Improvements:")
print(f"    v3 → v4 single best : {10.67 - best_single_v4:.2f}% absolute reduction")
print(f"    v3 → v4 ensemble    : {10.67 - ensemble_eer:.2f}% absolute reduction")
print(f"    Relative improvement : {(10.67 - ensemble_eer)/10.67*100:.1f}%")

print(f"\n  Model Details:")
v4_params = sum(p.numel() for p in LCNN_v4(N_FEAT_USED).parameters())
print(f"    v3: ~576K params | v4: ~{v4_params/1000:.0f}K params")
print(f"    Ensemble: {len(SEED_LIST)} × {v4_params/1000:.0f}K = ~{len(SEED_LIST)*v4_params/1000:.0f}K total")

print(f"\n  Output files:")
for seed in SEED_LIST:
    print(f"    best_lcnn_v4_seed{seed}.pt")
print(f"    v4_training_curves.png")
print(f"    features_v4/")

print("═"*65)
print("\n✓ All done!")